# Demo UI (Gradio) on Kaggle: Task, Replay and Live tabs (GPU for Live)

Starts `app/demo_app.py` with a temporary public link (`share=True`; the link is printed below and works while this
session runs). Task and Replay use the recorded data in the repo; Live loads the models on its first run and keeps
them loaded. To stop: interrupt the last cell.
Settings: GPU T4 x1 (two T4s if parser and agent are different large models), Internet on.

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
PARSER_MODEL = "Qwen/Qwen2.5-7B-Instruct"     # or "Qwen/Qwen2.5-1.5B-Instruct" with ADAPTER
ADAPTER = ""                                   # fine-tuned parser adapter folder (attach it as a Kaggle dataset)
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"
AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
BACKEND = "hf"
NEEDED = ["app/demo_app.py", "app/demo_core.py", "eval/demo.py"]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox / verifier / tools (own folder) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
if BACKEND == "vllm":
    subprocess.run(["pip", "install", "-q", "-r", "requirements-vllm.txt"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers; print('pandas', pandas.__version__, '| numpy', numpy.__version__, "
         "'| torch', torch.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
# ---------------- Gradio (installed next to the pinned packages) ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/demopkgs", "gradio>=5,<6"], check=True)
ENV_UI = dict(ENV, PYTHONPATH=f"{WORK}/evalpkgs:{WORK}/demopkgs", IGT_DEMO_PARSER=PARSER_MODEL, IGT_DEMO_ADAPTER=ADAPTER,
              IGT_DEMO_GENERATOR=GENERATOR_MODEL, IGT_DEMO_AGENT=AGENT_MODEL)

In [ ]:
# ---------------- Launch (the public link is printed; interrupt this cell to stop) ----------------
p = subprocess.Popen(["python", "app/demo_app.py", "--share", "--host", "0.0.0.0"], env=ENV_UI,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    p.terminate(); print("stopped")